# Manejo de datos RDF en Python con rdflib. Carga, edición, consulta

La librería más habitual para trabajar con RDF en Python es rdflib. Aunque existen alternativas, suelen construirse extendiendo rdflib de alguna manera (no es el caso de rudof).

En las siguientes líneas, símplemente cargaremos un grafo en memoria.

1º, clonamos este mismo repositorio para disponer de los ejemplos en el entorno de ejecución:

In [ ]:
!git clone https://github.com/cursosLabra/miw_websem.git

Tras ello, instalamos rdflib y sus dependencias con pip.

In [ ]:

!pip install rdflib

Y ahora lanzamos código para parsear con rdflib el grafo de datos situado en /Introduction/

In [ ]:

from rdflib import Graph
g = Graph()
base_path = "/content/miw_websem/Introduction/"
g.parse(base_path + "example1.ttl")
g.parse(base_path + "example2.ttl")
g.parse(base_path + "example3.ttl")
print(g.serialize(format="turtle"))

Una vez cargados los datos, vamos a ver algo de código para recorrerlos.

In [ ]:
# Los grafos de rdflib son iterables. Como tal, podemos recorrerlos con un bucle for each. Lo que tenemos en cada iteración es una tripleta

for a_triple in g:
  print(a_triple)

In [ ]:
# Las tripletas son tuplas de 3 elementos. Para los que no estéis muy habituados a trabajar en Python, una tupla es una lista inmutable. Las tuplas son, a su vez, iterables.
for a_triple in g:
  for elem in a_triple:
    print(elem)
  print("-----")

# Otra manera visual de recorrer las tripletas es usando unpacking de Python
for a_triple in g:
  s, p, o = a_triple
  print(s, p, o)


Graph de rdflib ofrece un método muy útil para recorrer tripletas y, potencialmente, hacer alguna consulta básica: Graph.triples(). Así funciona:

In [ ]:
# g.triples() es un método que espera 1 tupla de 3 elementos como parámetro. En caso de que esos 3 parámetros sean None, se comporta igual que al recorrer el grafo con un for each:
for a_triple in g.triples(  (None, None, None)  ):
  s, p, o = a_triple
  print(s,p,o)

In [ ]:
# El método gana "poder" cuando damos un valor a alguno de los elementos de esa tupla. Por ejemplo:
from rdflib import URIRef

for a_triple in g.triples(  (None, URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'), None)  ):
  s, p, o = a_triple
  print(s,p,o)

In [ ]:
# Con el código anterior hemos recorrido todas las tripletas del grafo que tienen rdf:type como predicado.
# Cada una de las posiciones de esa tupla sirve para filtrar, si queremos, tripletas que no coincidan con
# el valor que utilicemos.

# El primer elemento de la tupla fija un sujeto. Si es None, cualquier sujeto.
# El segundo elemento de la tupla fija un predicado. Si es None, cualquier predicado.
# El tercer elemento de la tupla fija un objeto. Si es None, cualquier objeto.

# Esto nos permite incluso hacer "consultas simples". No son consultas propiamente dichas,  en el sentido
# de que el resultado siempre son tripletas, estructura ?s, ?p ?o. No podemos sacar/introducir variables,
# pero puede resultar util

# Tripletas que contengan becarios:
for a_triple in g.triples(  (None,
                             URIRef('http://www.w3.org/1999/02/22-rdf-syntax-ns#type'),
                             URIRef('http://uniovi.es/Becario')   )  ):
  s, p, o = a_triple
  print(s,p,o)


In [9]:
# Tripletas  donde Ana sea sujeto:
for a_triple in g.triples(  (URIRef('http://uniovi.es/ana'),
                             None,
                             None   )  ):
  s, p, o = a_triple
  print(s,p,o)


http://uniovi.es/ana http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://uniovi.es/Profesor


In [ ]:
# Tripletas donde Ana sea objeto
for a_triple in g.triples(  (None,
                             None,
                             URIRef('http://uniovi.es/ana')   )  ):
  s, p, o = a_triple
  print(s,p,o)


rdflib también nos proporciona mecanismos para hacer consultas de SPARQL contra los datos cargados un en un objeto Graph(). Bastará con crear un string con una consulta SPARQL con sintaxis válida y llamar al método Graph.query().


In [ ]:
# Consulta para obtener pares instancia-clase
str_query = """
select * where {
  ?s a ?o
}
"""
query_result = g.query(str_query)

# Los resultados se recorren por filas. Quizá te recuerde a resultsets de librerías para ejecutar consultas SQL.
# Cada fila contiene objeto de dominio de rdflib (URIRef, Literal...) en variables que se llaman exactamente
# como los nombres de variables en la consulta SPARQL
for a_row in query_result:
  an_instance = a_row.s
  a_class = a_row.o
  print(an_instance, a_class)



In [ ]:
# Solamente instancias de profesor:
str_query = """
PREFIX uni: <http://uniovi.es/>

select * where {
  ?s a uni:Profesor
}
"""
query_result = g.query(str_query)

# Los resultados se recorren por filas, quizá te recuerde a resultsets de librerías para ejecutar consultas SQL.
# Cada fila contiene objeto de dominio de rdflib (URIRef, Literal...) en variables que se llaman exactamente como los nombres de variables en la consulta SPARQL
for a_row in query_result:
  print(a_row.s)


El código que hemos visto nos sirve para consultar datos del grafo, pero RDFlib también nos permite editar un grafo existente eliminando/añadiendo tripletas. Para hacer eso debemos de manejar, fundamentalmente, 3 clases:



*   URIRef --> Nos permite instanciar objetos que representan URIs
*   Literal --> Nos permite instancias cualquier tipo de literal. Los literales, además de su valor, pueden tener valores asociados como tipo de datos (string, int, float, date, bool, etc.), o etiqueta le lenguage (español, inglés, etc.).
*   BNode --> Nos permite instancias nodos anónimos. Mientras que conservemos en nuestro programa una referencia a estos objetos, podremos vinclular un mismo nodo anónimo con varios elementos.

No existe una clase para representar la idea de tripleta. Se añaden tripletas a un grafo mediante el método add, que espera recibir como parámetro una tupla conteniendo sujeto, predicado y objeto ordenados.




In [ ]:
from rdflib import Literal
# from rdflib import URIRef

juan = URIRef("http://uniovi.es/juan")
age = URIRef("http://xmlns.com/foaf/0.1/age")
edad_juan = Literal(21, datatype="http://www.w3.org/2001/XMLSchema#int")  # rdflib hace inferencias de tipo adecuadas
                                   # en general en función del tipo de datos de lo que se pase como primer valor al
                                   # constructor. Pero no viene mal declarar tipos explícitos y asegurarse de que
                                   # esto no fallará

g.add( (juan, age, edad_juan)  )
for a_triple in g.triples( (juan, None, None) ):
  s,p,o = a_triple
  print(s, p, o)

Hay una cuarta clase que no es esencial, pero nos ayuda mucho a escribir código de edición de grafos: Namespace(). Podemos declarar un espacio de nombres y añadir sujetos/propiades de manera mucho más sencilla:

In [ ]:
from rdflib import Namespace

otracosa = Namespace("http://otracosa.es/")
uni = Namespace("http://uniovi.es/")

# rdflib ya tiene algunos namespaces precargados típicos, que podemos importaqr isn necesidad de cargar de cero:

from rdflib import XSD

g.add(   (uni.ana, otracosa.cantidad_ojos, Literal(2,datatype=XSD.int))   )

for a_triple in g.triples( (uni.ana, None, None) ):
  s,p,o = a_triple
  print(s, p, o)


Si tenemos intención de serializar nuestro grafo, conviene hacer binding de los namespaces. Fíjate lo que ocurre con la lista de prefijos al tratar de serializar nuestro grafo actual:

In [ ]:
print(g.serialize())

rdflib asignará prefijos "guapos" a los namespaces que haya parseado o que ya conozca (ya conoce FOAF, por ejemplo). Pero aquellos que no, como http://otracosa.es/, les dará un valor por defecto, "feo".

Esto no tiene por qué ser así. Podemos indicar cómo queremos que serialice nuestros namespaces con prefijos que tengan sentido para nosotros.

In [ ]:
g.bind("o_c", otracosa)
print(g.serialize())

rdflib, no obstante, nos proporciona herramientas para manipular y recorrer rdflib.Graphs. No nos proporciona formas de consumir contenido de un endpoint SPARQL, que es un caso de uso habitual. Para eso, debemos utilizar la librería SPARQLWrapper. Es de los mismos desarrolladores y comparte algunos objetos de modelo.

In [ ]:
!pip install sparqlwrapper

In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

sparql = SPARQLWrapper("http://dbpedia.org/sparql")  # Construimos un objeto para acceder a cierto endpoint. En este caso, DBpedia
sparql.setQuery("""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

    SELECT ?label  WHERE {
      <http://dbpedia.org/resource/Oviedo> rdfs:label ?label
      }"""
    )  # Consulta para traer todas las etiquetas de Oviedo en DBpedia (potencvialmente en distintas lenguas)

sparql.setReturnFormat(JSON)  # Deberíamos configurar un formato de salida. Es habitual trabajar con JSON, pero tenemos más opciones
results = sparql.query().convert()  # Query hace la consulta, convert adapta los datos a nuestro formato de salida
for result in results["results"]["bindings"]:  # En el nodo results-->bindings del JSON encontramos una línea con los resultados.
    print(result["label"]["value"]) # En cada línea encontraremos claves que hacen referencia a las variables de nuestra query.
print("--------")
print(results["results"]["bindings"][0]) # Cada uno de esos nodos de resultado no contiene solo el valor sino más datos asociados, como tipo de datos o etiqueta de lenguaje.
                                         # No todos los bindings-->nombre de variable ofrecen los mismos campos. "value" siempre está, lo demás depende del tipo de datos que estés tratando.


In [ ]:


sparql = SPARQLWrapper("http://dbpedia.org/sparql")  # Construimos un objeto para acceder a cierto endpoint. En este caso, DBpedia
sparql.setQuery("""
    SELECT ?p ?o  WHERE {
      <http://dbpedia.org/resource/Oviedo> ?p ?o
      } LIMIT 100"""
    )  # Consulta para traer cosas sobre Oviedo (no más de 100)

sparql.setReturnFormat(JSON)
results = sparql.query().convert()
for result in results["results"]["bindings"]:
    print("Par predicado-objeto: " + result["p"]["value"] + " " + result["o"]["value"])
    print("Aspecto de la fila de resultado: " + str(result))
    print()

print("--------")
